In [1]:
import os
os.chdir("D:/tmp_workplace/dynamic_structure/250402/HELA_RIC_analysis/structure_predict/RPPH1")
import numpy as np
import sys
from scipy import sparse
from scipy.signal import convolve2d
import itertools
import math
from pathlib import Path

In [2]:
def dump_matrix(hicFile,chrom):
    title = hicFile.replace(".hic","_")
    res15 = title+"_"+"KR"+"_"+"15"+".matrix"
    res10 = title+"_"+"KR"+"_"+"10"+".matrix"
    res5 = title+"_"+"KR"+"_"+"5"+".matrix"
    res1 = title+"_"+"NONE"+"_"+"1"+".matrix"
    os.system("java -jar /storx/yxdeng/toatal_job/juicer_tools.jar dump observed KR %s %s %s BP 15 %s"%(hicFile,chrom,chrom,res15))
    os.system("java -jar /storx/yxdeng/toatal_job/juicer_tools.jar dump observed KR %s %s %s BP 10 %s"%(hicFile,chrom,chrom,res10))
    os.system("java -jar /storx/yxdeng/toatal_job/juicer_tools.jar dump observed KR %s %s %s BP 5 %s"%(hicFile,chrom,chrom,res5))
    os.system("java -jar /storx/yxdeng/toatal_job/juicer_tools.jar dump observed NONE %s %s %s BP 1 %s"%(hicFile,chrom,chrom,res1))
    return res15,res10,res5,res1

def get_oe(matrix):
    matrix = np.array(matrix).copy()
    lens = matrix.shape[0]
    mask = np.ones(lens)
    cut_off = lens * 0.0001
    matrix_num = matrix.copy()
    matrix_num[matrix_num > 0] = 1
    col_sum = matrix_num.sum(axis = 0)
    sites = np.arange(lens)
    for i in sites:
        if col_sum[i] <= cut_off:
            mask[i] = 0
    mask = mask == 1
    x_o = sites.copy()
    y_o = sites.copy()
    dis = np.zeros(lens)
    mask_o = mask.copy()
    exp_matrix = np.ones((lens, lens))
    x_e = sites.copy()
    y_e = sites.copy()
    for i in range(lens):
        diag_n = matrix[(x_o, y_o)]
        diag_line = diag_n[mask_o]
        if diag_line.shape[0] == 0:
            dis[i] = 0
        else :
            dis[i] = diag_line.mean()
            x_o = np.delete(x_o,-1)
            y_o = np.delete(y_o,0)
            mask_o = np.delete(mask_o,-1)
        
        if dis[i] == 0:
            x_e = np.delete(x_e,-1)
            y_e = np.delete(y_e,0)
            continue
        exp_matrix[(x_e, y_e)] = dis[i]
        exp_matrix[(y_e, x_e)] = dis[i]
        x_e = np.delete(x_e,-1)
        y_e = np.delete(y_e,0)

    oe = np.matrix(matrix/exp_matrix)
    return np.asarray(oe)

def get_matrix_npz(M1,M2,res):
    S1 = sparse.coo_matrix((M1['value'],(np.asarray(M1['S'], dtype= 'i4'),np.asarray(M1['E'], dtype= 'i4')))).toarray()
    S2 = sparse.coo_matrix((M2['value'],(np.asarray(M2['E'], dtype= 'i4'),np.asarray(M2['S'], dtype= 'i4')))).toarray()
    matrix = np.zeros((max(S1.shape),max(S1.shape)))
    matrix[np.nonzero(S1)] = S1[np.nonzero(S1)]
    matrix[np.nonzero(S2)] = S2[np.nonzero(S2)]
    if res == 10:
        W,H = matrix.shape
        for i in range(1,W-1):
            #matrix[i,i] = matrix[i-1,i-1] = matrix[i-1,i] = matrix[i-1,i+1] = matrix[i,i-1] = matrix[i,i+1] = matrix[i+1,i-1] = matrix[i+1,i] = matrix[i+1,i+1] = 0
            matrix[i,i] = matrix[i-1,i] = matrix[i,i-1] = matrix[i,i+1] = matrix[i+1,i] = 0
    return matrix

def extract_matrix(file_name,res):
    data = np.loadtxt(file_name, usecols=[0,1,2], dtype=[('S','<i4'),('E','<i4'),('value','<f8')])
    data['S'] = data['S'] / res
    data['E'] = data['E'] / res
    data['value'][data['value'] < 0] = 0
    data['value'][np.isnan(data['value'])] = 0
    mat = get_matrix_npz(data,data,res)
    oe = get_oe(mat)
    return mat,oe

def mask_diagonal(raw_matrix,oe_matirx):
    tmp = raw_matrix.copy()
    ones_bool = (oe_matirx <= 1)
    indices = np.where(ones_bool)
    indices
    for row,col in zip(indices[0],indices[1]):
        tmp[row,col] = 0
    return tmp

def gaussian(sigma,ksize):
    ksize = int(ksize)
    gaussian_k = np.zeros((ksize, ksize))
    for i in range(ksize):
        for j in range(ksize):
            gaussian_k[i, j] = np.round(math.exp((-1/(2*sigma**2))*(np.square(i-1) + np.square(j-1)))/(2*math.pi*sigma**2),4)
    sum = np.sum(gaussian_k)
    gaussian_k /= sum
    gaussian_k = np.round(gaussian_k,4)
    #print(gaussian_k)
    return gaussian_k

def gaussian_blur(Matrix,Kernel):
    convolved_matrix = convolve2d(Matrix, Kernel, mode='same', boundary='fill', fillvalue=0)
    return convolved_matrix

def NMS(M): 
    W, H = M.shape
    nms = np.copy(M)
    nms[0, :] = nms[W-1, :] = nms[:, 0] = nms[:, H-1] = 0
        
    for i in range(1, W-1):
        for j in range(1, H-1):
            if M[i, j] == 0:
                nms[i, j] = 0      
            else:
                if M[i, j] >= M[i-1, j-1] and M[i, j] >= M[i+1, j+1]:
                    nms[i, j] = M[i, j]
                else:
                    nms[i, j] = 0              
    return nms

def DT(NMS,res):
    W, H = NMS.shape
    DT = np.zeros([W, H])
    strong_edges = np.zeros([W, H])
    if res == 5:
        TL = 0.01 * np.max(NMS)
        TH = 0.03 * np.max(NMS)
    if res == 10:
        TL = 0.01 * np.max(NMS)
        TH = 0.03 * np.max(NMS)
    print(TL,TH)
    for i in range(1, W-1):
        for j in range(1, H-1):    
            if (NMS[i, j] > TH):
                strong_edges[i, j] = True
            else:
                strong_edges[i, j] = False
    last = np.sum(strong_edges)
    while(1):
        this = 0
        for i in range(1, W-1):
            for j in range(1, H-1):    
                if (strong_edges[i, j]):
                    DT[i, j] = True
                else:
                    if (NMS[i, j]>TL):
                        if ((DT[i-1, j-1:j+1].any()) or (DT[i+1, j-1:j+1].any()) or (DT[i, [j-1, j+1]].any())):
                            DT[i, j] = True
        this = np.sum(DT)
        if this - last == 0 :
            break
        last = this
    final_edges = DT.astype(int)    
    return final_edges

def matrix2_bedpe(M,output,res):
    ones_bool = (M == 1)
    #ones_bool = (M > 0)
    indices = np.where(ones_bool)
    files = open(output,'w')
    files.write('chrom1'+'\t'+'start1'+'\t'+'end1'+'\t'+'chrom2'+'\t'+'start2'+'\t'+'end2'+'\n')
    for row, col in zip(indices[0], indices[1]):
        row = int(row)
        col = int(col)
        row_start = str((row * res))
        row_end = str((row * res) + res)
        col_start = str((col * res))
        col_end = str((col * res) + res)
        strs = "NR_002312.1"+"\t"+row_start+"\t"+row_end+"\t"+"NR_002312.1"+"\t"+col_start+"\t"+col_end+"\n"
        files.write(strs)
    files.close()

def save_matrix_to_hic_pre(matrix,out_file):
    ones_bool = (matrix > 0)
    indices = np.where(ones_bool)
    files = open(out_file,'w')
    for row, col in zip(indices[0], indices[1]):
        if row > col :
            continue
        files.write("0"+"\t"+"NR_002312.1"+"\t"+str(int(row)*5)+"\t"+"0"+"\t"+"0"+"\t"+"NR_002312.1"+"\t"+str(int(col)*5)+"\t"+"1"+"\t"+str(int(float(matrix[row,col])))+"\n")
    files.close()

def canny(hic_matrix,res,oe_mask=False):
    kernel = gaussian(1.5,3)
    raw, oe = extract_matrix(hic_matrix,res)
    if oe_mask:
        ma = mask_diagonal(raw,oe)
        filer_m = gaussian_blur(ma,kernel)
        canny_out_title = hic_matrix.replace(".matrix",'') + "_" + str(res) + "_" + "OEmask" + ".bedpe"
    else:
        filer_m = gaussian_blur(raw,kernel)
        canny_out_title = hic_matrix.replace(".matrix",'') + "_" + str(res) + ".bedpe"
    if res == 10:
        dt = DT(filer_m,res)
    if res == 5:
        nms = NMS(filer_m)
        dt = DT(nms,res)  
    matrix2_bedpe(dt,canny_out_title,res)
    return canny_out_title

def matrix_10BP_to_matrix_5BP(inmatrix):
    im = open(inmatrix,"r")
    out_title = inmatrix.replace(".bedpe","") + "_remake10.bedpe"
    om = open(out_title,"w")
    om.write('chrom1'+'\t'+'start1'+'\t'+'end1'+'\t'+'chrom2'+'\t'+'start2'+'\t'+'end2'+'\n')
    tmp_set = set()
    for line in im:
        if 'chrom1' in line:
            continue
        sub = line.strip().split()
        s1 = int(sub[1])
        e1 = int(sub[2])
        s2 = int(sub[4])
        e2 = int(sub[5])
        #if (s1 % 10 == 5):
        #    s1 = s1 - 5
        #if (e1 % 10 == 5):
        #    e1 = e1 + 5
        #if (s2 % 10 == 5):
        #    s2 = s2 - 5
        #if (e2 % 10 == 5):
        #    e2 = e2 - 5
        tmp_set.add((s1,s1+5,s2,s2+5))
        tmp_set.add((s1,s1+5,e2-5,e2))
        tmp_set.add((e1-5,e1,s2,s2+5))
        tmp_set.add((e1-5,e1,e2-5,e2))
    for pixel in tmp_set:
        om.write("NR_002312.1"+"\t"+str(pixel[0])+"\t"+str(pixel[1])+"\t"+"NR_002312.1"+"\t"+str(pixel[2])+"\t"+str(pixel[3])+"\n")
    im.close()
    om.close()
    return out_title

def canny_diagonal_filling(in_matrix):
    tmp_matrix = np.zeros((70,70))
    inm = open(in_matrix,"r")
    out_matrix = in_matrix.replace(".bedpe","") + "_fill_diagonal.bedpe"
    outm = open(out_matrix,"w")
    outm.write('chrom1'+'\t'+'start1'+'\t'+'end1'+'\t'+'chrom2'+'\t'+'start2'+'\t'+'end2'+'\n')
    for line in inm:
        if 'chrom1' in line:
            continue
        sub = line.strip().split()
        tmp_matrix[int(int(sub[1])/5),int(int(sub[4])/5)] = 1
    for i in range(tmp_matrix.shape[0]):
        if (i - 1 >= 0):
            if (i + 1 <= 69):
                if (tmp_matrix[i-1,i+1] == 1) and (tmp_matrix[i+1,i-1] == 1):
                    tmp_matrix[i,i] = 1
            if (i + 2 <= 69):
                if (tmp_matrix[i-1,i+2] == 1) and (tmp_matrix[i+2,i-1] == 1):
                    tmp_matrix[i,i+1] = tmp_matrix[i+1,i] = 1
        if (i - 2 >= 0):
            if (i + 2 <= 69):
                if (tmp_matrix[i-2,i+2] == 1) and (tmp_matrix[i+2,i-2] == 1):
                    tmp_matrix[i-1,i+1] = tmp_matrix[i+1,i-1] = tmp_matrix[i,i] = 1
    ones_bool = (tmp_matrix == 1)
    indices = np.where(ones_bool)
    for row, col in zip(indices[0], indices[1]):
        outm.write("NR_002312.1"+"\t"+str(row*5)+"\t"+str(row*5+5)+"\t"+"NR_002312.1"+"\t"+str(col*5)+"\t"+str(col*5+5)+"\n")
    inm.close()
    outm.close()
    return out_matrix

def merge_canny_matrix(in_files_10,in_files_5):
    out_file = in_files_10.replace(".bedpe","") + "_merge_" + in_files_5
    out = open(out_file,"w")
    out.write('chrom1'+'\t'+'start1'+'\t'+'end1'+'\t'+'chrom2'+'\t'+'start2'+'\t'+'end2'+'\n')
    pixels_set = set()
    for infile in [in_files_10,in_files_5]:
        with open(infile,"r")as f:
            for line in f:
                if 'chrom1' in line:
                    continue
                sub = line.strip().split()
                pixels_set.add((sub[1],sub[2],sub[4],sub[5]))
    for pixel in pixels_set:
        s1,e1,s2,e2 = pixel
        out.write("NR_002312.1"+"\t"+str(s1)+"\t"+str(e1)+"\t"+"NR_002312.1"+"\t"+str(s2)+"\t"+str(e2)+"\n")
    out.close()
    return out_file

def canny_filter_gaussian(canny_result,rri_resulit):
    canny_filter_count = 0
    rri_filter_count = 0
    cannym = open(canny_result,"r")
    canny = np.zeros((70,70))
    for line in cannym:
        if 'chrom1' in line:
            continue
        sub = line.strip().split()
        canny[int(int(sub[1])/5),int(int(sub[4])/5)] = 1
        canny[int(int(sub[4])/5),int(int(sub[1])/5)] = 1
    cannym.close()
    rrim = open(rri_resulit,"r")
    rri = np.zeros((70,70))
    for line in rrim:
        sub = line.strip().split()
        if (sub[2] == 'NaN'):
            continue
        rri[int(int(sub[0])/5),int(int(sub[1])/5)] = float(sub[2])
        rri[int(int(sub[1])/5),int(int(sub[0])/5)] = float(sub[2])
    rrim.close()
    kernel = gaussian(1.5,3)
    rri_gaussian = gaussian_blur(rri,kernel)
    rri_gaussian_max = np.max(rri_gaussian)
    #print('rri_gaussian_max',rri_gaussian_max)
    #rri_cutoff = np.percentile(rri_gaussian, 5) 
    rri_cutoff = rri_gaussian_max * 0.01
    print('rri_cutoff',rri_cutoff)
    tmp_matrix = np.zeros((70,70))
    for row in range(canny.shape[0]):
        for col in range(canny.shape[1]):
            if (canny[row,col] == 1):
                if (float(rri_gaussian[row,col]) > float(rri_cutoff)):
                    tmp_matrix[row,col] = 1
                else:
                    rri_filter_count += 1
                if (float(rri[row,col]) <= 0):
                    tmp_matrix[row,col] = 0
            else:
                canny_filter_count += 1
    outmatrix = canny_result.replace(".bedpe","") + "_filter_gaussian.bedpe"
    out = open(outmatrix,"w")
    out.write('chrom1'+'\t'+'start1'+'\t'+'end1'+'\t'+'chrom2'+'\t'+'start2'+'\t'+'end2'+'\n')
    ones_bool = (tmp_matrix > 0)
    indices = np.where(ones_bool)
    for row,col in zip(indices[0],indices[1]):
        out.write("NR_002312.1"+"\t"+str(row*5)+"\t"+str(row*5+5)+"\t"+"NR_002312.1"+"\t"+str(col*5)+"\t"+str(col*5+5)+"\n")
    out.close()
    #print('rri_filter_count',rri_filter_count)
    #print('canny_filter_count',canny_filter_count)
    return outmatrix


In [ ]:
hic_file = ''
res15_matrix,res10_matrix,res5_matrix,res1_matrix = dump_matrix(hic_file,'NR_002312.1')

In [3]:
res5_matrix = 'RPPH1_5_VCRT.matrix'
res10_matrix = 'RPPH1_10_KR.matrix'

In [4]:
#dump_matrix(".hic",'NR_002312.1')
canny_out_10_file = canny(res10_matrix,10,oe_mask=False)
canny_out_5_file = canny(res5_matrix,5,oe_mask=True)

0.125534819182 0.37660445754599997
0.24088022869200004 0.7226406860760001


In [5]:
remake_file = matrix_10BP_to_matrix_5BP(canny_out_10_file)
filling_file = canny_diagonal_filling(canny_out_5_file)
merge_file = merge_canny_matrix(remake_file,filling_file)

In [6]:
gaussian_filter_file = canny_filter_gaussian(merge_file,res5_matrix)
#filter_file = canny_filter(merge_file,res10_matrix)

rri_cutoff 0.24088022869200004
